In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from ocr_vs_vlm.results_final.shared.colors import MODEL_COLORS, get_model_color, get_dataset_color, APPROACH_COLORS
from ocr_vs_vlm.results_final.shared.stats_utils import bootstrap_ci, wilcoxon_test, cohens_d
from ocr_vs_vlm.results_final.shared.viz_utils import setup_plotly_template, create_metric_boxplot
from ocr_vs_vlm.results_final.shared.data_loader import load_experiment_data, extract_models_from_columns

setup_plotly_template()
print("✓ Setup complete")

## 1. Load Parsing Experiment Data

In [ ]:
PARSING_PHASES = ['phase_1', 'phase_2', 'phase_3']
DATASETS = ['IAM_mini', 'ICDAR_mini', 'VOC2007']

all_data = []
for dataset in DATASETS:
    for phase in PARSING_PHASES:
        try:
            df = load_experiment_data(dataset, phase)
            all_data.append(df)
            print(f"✓ {dataset}/{phase}: {len(df)} samples")
        except FileNotFoundError:
            # Try phase_3a variant
            try:
                df = load_experiment_data(dataset, phase.replace('phase_3', 'phase_3a'))
                df['phase'] = phase  # Normalize
                all_data.append(df)
                print(f"✓ {dataset}/{phase} (as 3a): {len(df)} samples")
            except FileNotFoundError:
                print(f"✗ Missing {dataset}/{phase}")

df_parsing = pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()
print(f"\n📊 Total: {len(df_parsing)} samples")

## 2. OCR vs VLM Comparison (CER)

In [ ]:
# Extract CER scores
models = extract_models_from_columns(df_parsing)

cer_by_approach = {'OCR (P1)': [], 'VLM Generic (P2)': [], 'VLM Task-aware (P3)': []}
approach_map = {'phase_1': 'OCR (P1)', 'phase_2': 'VLM Generic (P2)', 'phase_3': 'VLM Task-aware (P3)'}

stats_rows = []
for phase in PARSING_PHASES:
    phase_df = df_parsing[df_parsing['phase'] == phase]
    approach = approach_map[phase]
    
    for model in models:
        col = f'cer_{model}'
        if col not in phase_df.columns:
            col = f'prediction_{model}'  # Fallback - would need metric calculation
            continue
        
        scores = phase_df[col].dropna().values
        if len(scores) > 0:
            mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
            stats_rows.append({
                'Approach': approach,
                'Model': model,
                'CER': mean,
                'CI_Lower': ci_lo,
                'CI_Upper': ci_hi
            })

stats_df = pd.DataFrame(stats_rows)
if len(stats_df) > 0:
    display(stats_df.pivot_table(index='Model', columns='Approach', values='CER').round(4))

In [ ]:
# Visualization: CER by approach
if len(stats_df) > 0:
    fig = go.Figure()
    
    for approach in stats_df['Approach'].unique():
        app_data = stats_df[stats_df['Approach'] == approach]
        color = APPROACH_COLORS.get('ocr_baseline' if 'OCR' in approach else 'vlm_generic', '#6B7280')
        
        fig.add_trace(go.Bar(
            x=app_data['Model'],
            y=app_data['CER'],
            name=approach,
            marker_color=color,
            error_y=dict(type='data', array=app_data['CER'] - app_data['CI_Lower'])
        ))
    
    fig.update_layout(
        title='Parsing: CER by Approach (Lower is Better)',
        yaxis_title='Character Error Rate (CER)',
        barmode='group',
        height=500
    )
    fig.show()
else:
    print("⚠️ No CER data available - check column naming in results files")

## 3. Dataset Comparison

In [ ]:
# Compare across datasets
dataset_stats = []
for dataset in DATASETS:
    ds_df = df_parsing[df_parsing['dataset'] == dataset]
    for model in models:
        for metric in ['cer', 'wer']:
            col = f'{metric}_{model}'
            if col in ds_df.columns:
                scores = ds_df[col].dropna().values
                if len(scores) > 0:
                    mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                    dataset_stats.append({
                        'Dataset': dataset,
                        'Model': model,
                        'Metric': metric.upper(),
                        'Value': mean
                    })

ds_df = pd.DataFrame(dataset_stats)
if len(ds_df) > 0:
    pivot = ds_df.pivot_table(index=['Dataset', 'Model'], columns='Metric', values='Value')
    display(pivot.round(4))

## 4. Summary

In [ ]:
print("=" * 70)
print("PARSING EXPERIMENTS SUMMARY")
print("=" * 70)

if len(stats_df) > 0:
    # Best by approach
    print("\n📊 Best CER by Approach:")
    for approach in stats_df['Approach'].unique():
        best = stats_df[stats_df['Approach'] == approach].nsmallest(1, 'CER').iloc[0]
        print(f"   {approach}: {best['Model']} → CER={best['CER']:.4f}")
    
    # Overall best
    overall_best = stats_df.nsmallest(1, 'CER').iloc[0]
    print(f"\n🏆 Overall Best: {overall_best['Model']} ({overall_best['Approach']}) → CER={overall_best['CER']:.4f}")

print("\n" + "=" * 70)